In [310]:
import pandas as pd
import numpy as np

In [311]:
bm_raw = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/bm_raw_new.csv")
bss_pca = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/bss_pca_raw.csv")
pca_sellers = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/PCA for Sellers.csv")
classification = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/classification_mapping.xlsx")
Brand_mapping = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/Brand_mapping.xlsx")


In [312]:
bss_redemption = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Redemption Reports/BSS Redemption.csv")

In [313]:
free_credit = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Redemption Reports/Free-Credit-Data-Report_2026-01-14.csv")

In [314]:
redemption_Consumption = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Redemption Reports/Redemption-Consumption-Data-Report_2026-01-14.csv")

In [315]:
mtd_brand = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Redemption Reports/MTDOverBurnBrand.csv")

In [316]:
mtd_seller = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Redemption Reports/MTDOverBurnSeller.csv")

In [442]:
input_sheet = pd.read_excel(r"/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/Input sheet.xlsx")

# BM Raw

In [317]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id',
       'adgroup_id', 'brand', 'commodity_name', 'analytic_vertical',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'Alpha_MP', 'BMP_Tag', 'Team', 'budget_type',
       'cost_model', 'pacing', 'campaign_type', 'campaign_name', 'status',
       'start_date', 'end_date', 'commodity_id', 'Channel', 'allocated_budget',
       'gross_budget', 'uniqueviewcount', 'actioncount', 'total_cost_x',
       'addtobasketcount', 'viewcount', 'cnt_lid', 'attr_units',
       'attr_revenue', 'overburn_share', 'Actual_spend', 'Budget_adgroup',
       'ROI', 'CTR', 'CVR'],
      dtype='object')

In [318]:
bm_raw = bm_raw[bm_raw['total_cost_x']>5]

In [319]:
# bss_pca.columns

In [320]:
#  pca_sellers.columns

In [321]:
classification.columns

Index(['Super Categories_Tag', 'BU_Tag', 'Wrong Tagging', 'New Tag', 'Type',
       'Adv. Name', 'Tag', 'Remarks', 'Super Categories_HL', 'BU_HL', 'BU_HL2',
       'HL Supercat'],
      dtype='object')

In [322]:
Brand_mapping.columns

Index(['Incorrect Account Name', 'Correct Name', 'BU', 'Concatenate',
       'Advertiser Name', 'Tag', 'Ad Account ID', 'Ad Account Name',
       'Business Account ID', 'Business Account Name', 'Vertical',
       'Current SC', 'New SC'],
      dtype='object')

- New Alpha/MP

In [323]:
incorrect_brands = Brand_mapping['Ad Account ID'].unique()

bm_raw['New Alpha/MP'] = np.where(
    bm_raw['advertiser_id'].isin(incorrect_brands) | (bm_raw['Team'] == 'IC'),
    'IC',
    bm_raw['Alpha_MP']
)


In [324]:
bm_raw['New Alpha/MP'].value_counts()

,count
New Alpha/MP,
MP,452982
Alpha,42970
IC,2047


- FCFF Alpha/MP

In [325]:
bm_raw['FCFF Alpha/MP'] = np.where(bm_raw['marketplace'] == "HYPERLOCAL","HL",bm_raw['Alpha_MP'])

In [326]:
bm_raw['FCFF Alpha/MP'].value_counts()

,count
FCFF Alpha/MP,
MP,454825
Alpha,36818
HL,6356


- New Supercat

In [327]:
brand_map = dict(zip(Brand_mapping['Vertical'], Brand_mapping['New SC']))
sc_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

conditions = [
    bm_raw['marketplace'] == "Grocery",
    bm_raw['analytic_vertical'].isin(brand_map.keys()),
    bm_raw['analytic_super_category'].isin(brand_map.keys())
]

choices = [
    "Grocery",
    bm_raw['analytic_vertical'].map(brand_map),
    bm_raw['analytic_super_category'].map(brand_map)
]

fallback = bm_raw['analytic_super_category'].map(sc_map).fillna(bm_raw['analytic_super_category'])

bm_raw['New Supercat'] = np.select(conditions, choices, default=fallback)


In [328]:
# print(bm_raw['New Supercat'].value_counts().to_string())


- Supercategory matching with FCFF's Super category

In [329]:
bm_raw['SC match FCFF'] = bm_raw['New Supercat'].isin(classification['Super Categories_Tag'])

In [330]:
# bm_raw['SC match FCFF'].value_counts()

- New BU

In [331]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bm_raw['New BU'] = bm_raw['New Supercat'].map(sc_bu_map)

In [332]:
print(bm_raw['New BU'].isnull().sum())

289


- Spend

In [333]:
bm_raw['Spends'] = bm_raw['total_cost_x']

In [334]:
bm_raw['Spends'].sum()

np.float64(1159516729.84)

- Brand

In [335]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bm_raw['lookup_key'] = bm_raw['advertiser_name'].astype(str) + bm_raw['analytic_super_category'].astype(str)

cond_a = bm_raw['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bm_raw['analytic_super_category'].isin(Brand_mapping['BU'])

bm_raw['Brand'] = np.where(
    cond_a & cond_b,
    bm_raw['advertiser_name'].map(brand_map),
    bm_raw['brand']
)
bm_raw['Brand'] = bm_raw['Brand'].fillna(bm_raw['brand'])


# BSS PCA

In [336]:
bss_pca.columns

Index(['Campaign ID', 'Marketplace', 'BU', 'Supercategory', 'HL_BU', 'store',
       'analytic_super_category', 'Brands', 'Store Name',
       'creative_template_id', 'creative_type', 'costmodel', 'Ad Account ID',
       'Ad Account Name', 'Business Account ID', 'Business Account Name',
       'Team', 'Alpha_MP', 'BMP_Tag', 'Channel', 'lifestyle_supercategory',
       'spend', 'views', 'clicks', 'ppv', 'click_direct_units',
       'click_indirect_units', 'click_direct_revenue',
       'click_indirect_revenue', 'Product', 'Average CPC', 'CTR', 'ROI'],
      dtype='object')

In [337]:
bss_pca = bss_pca.copy()

- New Alpha/MP

In [338]:
incorrect_brands = Brand_mapping['Advertiser Name'].unique()

bss_pca['New Alpha/MP'] = np.where(
    bss_pca['Business Account Name'].isin(incorrect_brands) | (bss_pca['Team'] == 'IC'),
    'IC',
    bss_pca['Alpha_MP']
)


- New Supercat

In [339]:
class_map = classification.drop_duplicates('Wrong Tagging').set_index('Wrong Tagging')['New Tag'].to_dict()

conditions = [ bss_pca['BU'] == "Grocery", bss_pca['BU'] == "Mobile", bss_pca['Supercategory'] == "PetCare", bss_pca['Supercategory'].isin(class_map.keys()) ]
choices = [ "Grocery", "Mobile", "Pets", bss_pca['Supercategory'].map(class_map) ]

bss_pca['New Supercat'] = np.select(conditions, choices, default=bss_pca['Supercategory'])


In [340]:
bss_pca['New Supercat'].isnull().sum()

np.int64(0)

- Supercategory matching with FCFF's Super category

In [341]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [342]:
bss_pca['SC match FCFF'].value_counts()

,count
SC match FCFF,
True,318830
False,5826


- New BU

In [343]:
sc_bu_map = classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

- Brand

In [344]:
brand_map = Brand_mapping.drop_duplicates('Incorrect Account Name').set_index('Incorrect Account Name')['Correct Name'].to_dict()

bss_pca['lookup_key'] = bss_pca['Ad Account Name'].astype(str) + bss_pca['Supercategory'].astype(str)

cond_a = bss_pca['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bss_pca['Supercategory'].isin(Brand_mapping['BU'])

bss_pca['Brand'] = np.where(
    cond_a & cond_b,
    bss_pca['Ad Account Name'].map(brand_map),
    bss_pca['Brands']
)
bss_pca['Brand'] = bss_pca['Brand'].fillna(bss_pca['Brands'])

- FCFF Alpha/MP

In [345]:
bss_pca['FCFF Alpha/MP'] = np.where(bss_pca['Alpha_MP'] == "IC","Alpha",np.where(bss_pca['Alpha_MP'] =="3P","MP",bss_pca['Alpha_MP']))

In [346]:
bss_pca['BUxBrand'] = bss_pca['New BU'].astype(str) + bss_pca['Brands'].astype(str)
lookup_map = bss_pca.drop_duplicates('BUxBrand').set_index('BUxBrand')['New Supercat'].to_dict()

bss_pca['Check'] = bss_pca['BUxBrand'].map(lookup_map)

In [347]:
bss_pca[bss_pca['New BU'] == 0 ]['spend'].sum()

np.float64(0.0)

- check and redo the Supercat and BU Mapping

In [348]:
bss_pca['New Supercat'] = np.where(bss_pca['SC match FCFF'] == False,bss_pca['Check'], bss_pca['New Supercat'])

In [349]:
sc_bu_map = classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

In [350]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [351]:
bss_pca[bss_pca['SC match FCFF'] == False]['spend'].sum()

np.float64(231239.74)

# PCA for Sellers

In [352]:
pca_sellers.columns

Index(['campaign_id', 'seller_id', 'BU', 'Supercategory', 'Store Name',
       'analytic_super_category', 'lifestyle_supercategory', 'brand',
       'KAM/NKAM', 'BMP_Tag', 'Channel', 'status', 'adspends', 'views',
       'clicks', 'revenue', 'units', 'CTR', 'CVR', 'ROI'],
      dtype='object')

In [353]:
pca_seller = pca_sellers.copy()

- Alpha/MP

In [354]:
pca_seller['Alpha/MP'] = "MP"

- Campaign Type

In [355]:
pca_seller['Campaign Type'] = "PCA"

- New Supercatagory

In [356]:
sc_map_cls = classification.drop_duplicates('Wrong Tagging').set_index('Wrong Tagging')['New Tag'].to_dict()

pca_seller['New Supercatagory'] = pca_seller['Supercategory'].map(sc_map_cls).fillna(pca_seller['Supercategory'])

- Supercategory matching with FCFF's Super category

In [357]:
pca_seller['SC match FCFF'] = pca_seller['New Supercatagory'].isin(classification['Super Categories_Tag'])

In [358]:
pca_seller['SC match FCFF'].value_counts()

,count
SC match FCFF,
True,298156
False,7707


In [359]:
print(round(pca_seller[pca_seller['SC match FCFF'] == True]['adspends'].sum()))

9361205


- BU

In [360]:
sc_bu_map =classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()
pca_seller['BU'] = pca_seller['New Supercatagory'].map(sc_bu_map)

In [361]:
#  pca_seller['BU'].value_counts()

In [362]:
brand_sc_map =pca_seller.drop_duplicates('brand').set_index('brand')['Supercategory'].to_dict()
pca_seller['Check'] = pca_seller['brand'].map(brand_sc_map)

- Check and redo Supercat and BU mapping

In [363]:
pca_seller['New Supercatagory'] = np.where(
    (pca_seller['SC match FCFF'] == False) & (pca_seller['New Supercatagory'] != 'Not assigned'),
    pca_seller['Check'],
    pca_seller['New Supercatagory']
)


In [364]:
sc_bu_map =classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()
pca_seller['BU'] = pca_seller['New Supercatagory'].map(sc_bu_map)

In [365]:
pca_seller['SC match FCFF'] = pca_seller['New Supercatagory'].isin(classification['Super Categories_Tag'])

In [366]:
print(round(pca_seller[pca_seller['SC match FCFF'] == True]['adspends'].sum()))

9457362


# MTD Overburn Brand and Sellers

In [367]:
brand_mtd = mtd_brand[["CampaignId","BudgetType","CampaignType","Overburn"]].copy()

In [368]:
seller_mtd = mtd_seller[["CampaignId","BudgetType","CampaignType","Overburn"]].copy()

In [369]:
mtd_overburn = pd.concat([brand_mtd,seller_mtd],axis=0)


In [370]:
print(round(mtd_overburn['Overburn'].sum()))

1746457


In [371]:
mtd_overburn.columns

Index(['CampaignId', 'BudgetType', 'CampaignType', 'Overburn'], dtype='object')

# BSS Rredemption

In [372]:
bss_redemption.columns

Index(['campaign_id', 'upi_id', 'billing_period_hash', 'payment_type',
       'account_id', 'campaign_spend', 'campaign_redeem_fc', 'campaign_redeem',
       'campaign_name', 'campaign_start_date', 'campaign_end_date',
       'campaign_budget', 'campaign_type', 'budget_type', 'market_place',
       'cost_model', 'brands', 'super_category_meta'],
      dtype='object')

- Filtered BSS Redemption Report

In [373]:
bss_redem = bss_redemption[bss_redemption['billing_period_hash'].astype(str).str.startswith('2026-01')].copy()


In [374]:
round(bss_redem['campaign_spend'].sum())

668919999

In [375]:
round(bss_redem['campaign_redeem_fc'].sum())

85172020

In [376]:
round(bss_redem['campaign_redeem'].sum())

583747979

In [377]:
round(bss_redem.groupby("campaign_type")['campaign_redeem'].sum())

,campaign_redeem
campaign_type,
BRAND_DISPLAY_ADS,65177.0
BRAND_PCA,72004791.0
BRAND_PLA,511678010.0


# Seller Portal Data

- Redemption&Consumption and FreeCredit Seller Data

In [378]:
free_credit.columns

Index(['campaign_id', 'seller_id', 'amount', 'date', 'merchant_id',
       'compartment_type'],
      dtype='object')

In [379]:
redemption_Consumption.columns

Index(['campaign_id', 'seller_id', 'amount', 'date', 'merchant_id'], dtype='object')

In [380]:
fc = free_credit.copy()
rc = redemption_Consumption.copy()

- Subtracted FC amount from RC amount to get Net FC

In [381]:
rc_unique = rc.groupby(['campaign_id', 'seller_id'])['amount'].sum().reset_index()
fc_unique = fc.groupby(['campaign_id', 'seller_id'])['amount'].sum().reset_index()

df_final = rc_unique.merge(fc_unique, on=['campaign_id', 'seller_id'], how='outer', suffixes=('_rc', '_fc'))

df_final['amount_rc'] = df_final['amount_rc'].fillna(0)
df_final['amount_fc'] = df_final['amount_fc'].fillna(0)

df_final['Net_amount'] = df_final['amount_rc'] - df_final['amount_fc']

rc_final = df_final[['campaign_id', 'seller_id', 'amount_rc', 'amount_fc', 'Net_amount']].copy()


In [382]:
round(rc_final['amount_rc'].sum())

413402776

In [383]:
round(rc_final['amount_fc'].sum())

71398423

In [384]:
round(rc_final['Net_amount'].sum())

342004353

In [385]:
rc_final.columns


Index(['campaign_id', 'seller_id', 'amount_rc', 'amount_fc', 'Net_amount'], dtype='object')

# Search Allocation

1. BM RAW


- BM Raw matched with FCFF Mapping

In [386]:
bm_raw_df = bm_raw[bm_raw['SC match FCFF'] == True][[ 'campaign_id', 'seller_id','FCFF Alpha/MP','New BU',
                    'analytic_vertical','campaign_type', 'Brand', 'Spends',
                     'Actual_spend','New Supercat','Team', 'BMP_Tag','advertiser_id' ]]

In [387]:
bm_raw_df = bm_raw_df.rename(columns={'FCFF Alpha/MP':'Alpha/MP','Team':'KAM/NKAM'})

In [388]:
bm_raw_df.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'New BU', 'analytic_vertical',
       'campaign_type', 'Brand', 'Spends', 'Actual_spend', 'New Supercat',
       'KAM/NKAM', 'BMP_Tag', 'advertiser_id'],
      dtype='object')

- PCA Sellers matched with FCFF Mapping

In [389]:
pca_seller_df = pca_seller[pca_seller['SC match FCFF'] == True] \
                          [['campaign_id', 'seller_id','Alpha/MP', 'BU','Campaign Type',
                          'brand','adspends', 'New Supercatagory', 'KAM/NKAM','BMP_Tag' ]]

In [390]:
pca_seller_df['Actual_spend'] = pca_seller_df['adspends']

In [391]:
pca_seller_df['analytic_vertical'] = np.nan

In [392]:
pca_seller_df['AdvertiserID'] = np.nan

In [393]:
pca_seller_df.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'BU', 'Campaign Type', 'brand',
       'adspends', 'New Supercatagory', 'KAM/NKAM', 'BMP_Tag', 'Actual_spend',
       'analytic_vertical', 'AdvertiserID'],
      dtype='object')

In [394]:
pca_seller_df = pca_seller_df.rename(columns={'campaign_id':'campaign_id', 'seller_id':'seller_id', 'Alpha/MP':'Alpha/MP', 'BU':'New BU', 'Campaign Type':'campaign_type', 'brand':'Brand',
       'adspends':'Spends', 'New Supercatagory':'New Supercat', 'KAM/NKAM':'KAM/NKAM', 'BMP_Tag':'BMP_Tag', 'Actual_spend':'Actual_spend',
       'analytic_vertical':'analytic_vertical', 'AdvertiserID':'advertiser_id'})

In [395]:
pca_seller_df.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'New BU', 'campaign_type',
       'Brand', 'Spends', 'New Supercat', 'KAM/NKAM', 'BMP_Tag',
       'Actual_spend', 'analytic_vertical', 'advertiser_id'],
      dtype='object')

- concatenate bm_raw and pca_seller files to get "Final BM Raw"

In [396]:
bm_raw_final = pd.concat([bm_raw_df,pca_seller_df],axis=0)

In [397]:
bm_raw_final.columns

Index(['campaign_id', 'seller_id', 'Alpha/MP', 'New BU', 'analytic_vertical',
       'campaign_type', 'Brand', 'Spends', 'Actual_spend', 'New Supercat',
       'KAM/NKAM', 'BMP_Tag', 'advertiser_id'],
      dtype='object')

In [398]:
# bm_raw_final['advertiser_id'].value_counts()

In [399]:
round(bm_raw_final['Spends'].sum())

1168837904

- Campaign Type

In [400]:
bm_raw_final['Type'] = np.where(bm_raw_final['campaign_type'] == "PCA","PCA",bm_raw_final['campaign_type'])

In [401]:
bm_raw_final['campaign_type'] = np.where(bm_raw_final['Type'] == "PCA","PLA",bm_raw_final['campaign_type'])

- Shopsy

In [402]:
bm_raw_final['Shopsy'] = np.where(bm_raw_final['New Supercat'].isin(["ShopsyLifeStyle","ShopsyBGM","ShopsyHome","ShopsyElectronics","ShopsyLarge","ShopsyFurniture"]),"Shopsy","Flipkart")

- Seller Portal Redemption

In [403]:
SP_sums = rc_final.groupby(['campaign_id', 'seller_id'])['Net_amount'].sum()

bm_raw_final['mapped_SP_Net'] = (
    bm_raw_final.set_index(['campaign_id', 'seller_id'])
    .index.map(SP_sums)
    .fillna(0)
)

denominator = bm_raw_final.groupby(['campaign_id', 'seller_id', 'campaign_type'])['Spends'].transform('sum')

bm_raw_final['Seller Portal Redemption'] = np.where(
    bm_raw_final['campaign_type'] == "PLA",
    (bm_raw_final['mapped_SP_Net'] * bm_raw_final['Spends']) / denominator,
    0
)

bm_raw_final['Seller Portal Redemption'] = bm_raw_final['Seller Portal Redemption'].replace([np.inf, -np.inf], 0).fillna(0)


In [404]:
bm_raw_final['Seller Portal Redemption'].sum()

np.float64(341928160.4617001)

- converted campaignid and accountid to uppercase to match between bss redemption and bm raw final dataframes

In [405]:
bss_redem['campaign_id'] = bss_redem['campaign_id'].astype(str).str.upper().str.strip()
bss_redem['account_id'] = bss_redem['account_id'].astype(str).str.upper().str.strip()

bm_raw_final['campaign_id'] = bm_raw_final['campaign_id'].astype(str).str.upper().str.strip()
bm_raw_final['advertiser_id'] = bm_raw_final['advertiser_id'].astype(str).str.upper().str.strip()


- BSS Redemption

In [406]:
bss_totals = bss_redem.groupby(['campaign_id', 'campaign_type', 'account_id'])['campaign_redeem'].sum()
bss_totals.index.names = ['campaign_id', 'campaign_type', 'advertiser_id']

bm_raw_map = bm_raw_final.set_index(['campaign_id', 'campaign_type', 'advertiser_id']).index
bm_raw_final['BSS_Campaign_redeem'] = bm_raw_map.map(bss_totals).fillna(0)


denominator = bm_raw_final.groupby(['campaign_id', 'campaign_type', 'advertiser_id'])['Actual_spend'].transform('sum')

bm_raw_final['BSS Redemption'] = np.where(
    (bm_raw_final['campaign_type'] == "BRAND_PLA") & (denominator > 0),
    (bm_raw_final['BSS_Campaign_redeem'] * bm_raw_final['Actual_spend']) / denominator,
    0
)


In [407]:
round(bm_raw_final['BSS Redemption'].sum())

511674089

- Free Credit for Brand_PLA

In [408]:
bss_totals = bss_redem.groupby(['campaign_id', 'campaign_type', 'account_id'])['campaign_redeem_fc'].sum()
bss_totals.index.names = ['campaign_id', 'campaign_type', 'advertiser_id']

bm_raw_map = bm_raw_final.set_index(['campaign_id', 'campaign_type', 'advertiser_id']).index
bm_raw_final['BSS_Campaign_redeem'] = bm_raw_map.map(bss_totals).fillna(0)


denominator = bm_raw_final.groupby(['campaign_id', 'campaign_type', 'advertiser_id'])['Spends'].transform('sum')

bm_raw_final['Free Credit BPLA'] = np.where(
    (bm_raw_final['campaign_type'] == "BRAND_PLA") & (denominator > 0),
    (bm_raw_final['BSS_Campaign_redeem'] * bm_raw_final['Spends']) / denominator,
    0
)


In [409]:
round(bm_raw_final['Free Credit BPLA'].sum())

71612008

- Free Credit for PLA

In [410]:
SP_sums = rc_final.groupby(['campaign_id', 'seller_id'])['amount_fc'].sum()

bm_raw_final['mapped_SP_Net'] = (
    bm_raw_final.set_index(['campaign_id', 'seller_id'])
    .index.map(SP_sums)
    .fillna(0)
)

denominator = bm_raw_final.groupby(['campaign_id', 'seller_id', 'campaign_type'])['Spends'].transform('sum')

bm_raw_final['Free Credit PLA'] = np.where(
    bm_raw_final['campaign_type'] == "PLA",
    (bm_raw_final['mapped_SP_Net'] * bm_raw_final['Spends']) / denominator,
    0
)

bm_raw_final['Free Credit PLA'] = bm_raw_final['Free Credit PLA'].replace([np.inf, -np.inf], 0).fillna(0)


In [411]:
round(bm_raw_final['Free Credit PLA'].sum())

61373583

converted campaignid and campaigntype to uppercase to get exact match

In [412]:
mtd_overburn['CampaignId'] = mtd_overburn['CampaignId'].astype(str).str.upper().str.strip()
mtd_overburn['CampaignType'] = mtd_overburn['CampaignType'].astype(str).str.upper().str.strip()

- OverBurn for Brand_PLA

In [413]:
overburn_totals = mtd_overburn.groupby(['CampaignId', 'CampaignType'])['Overburn'].sum()

bm_raw_final['mapped_Overburn'] = (
    bm_raw_final.set_index(['campaign_id', 'campaign_type'])
    .index.map(overburn_totals)
    .fillna(0)
)

denominator = bm_raw_final.groupby(['campaign_id', 'campaign_type'])['Spends'].transform('sum')

bm_raw_final['Overburn_BPLA'] = np.where(
    (bm_raw_final['campaign_type'] == "BRAND_PLA") & (denominator > 0),
    (bm_raw_final['mapped_Overburn'] * bm_raw_final['Spends']) / denominator,
    0
)


In [414]:
bm_raw_final['Overburn_BPLA'].sum()

np.float64(702466.24)

- OverBurn for PLA

In [415]:
overburn_totals = mtd_overburn.groupby(['CampaignId', 'CampaignType'])['Overburn'].sum()

bm_raw_final['mapped_Overburn'] = (
    bm_raw_final.set_index(['campaign_id', 'campaign_type'])
    .index.map(overburn_totals)
    .fillna(0)
)

denominator = bm_raw_final.groupby(['campaign_id', 'campaign_type'])['Spends'].transform('sum')

bm_raw_final['Overburn_PLA'] = np.where(
    (bm_raw_final['campaign_type'] == "PLA") & (denominator > 0),
    (bm_raw_final['mapped_Overburn'] * bm_raw_final['Spends']) / denominator,
    0
)


In [416]:
round(bm_raw_final['Overburn_PLA'].sum())

1043645

- Net Spends

In [417]:
bm_raw_final['Net Spend'] = bm_raw_final['Spends'] - bm_raw_final['Free Credit BPLA'] - bm_raw_final['Free Credit PLA'] - bm_raw_final['Overburn_BPLA'] - bm_raw_final['Overburn_PLA']

In [418]:
round(bm_raw_final['Net Spend'].sum())

1034106201

- Final Spends

In [419]:
bm_raw_final['Final_Spends'] = bm_raw_final['BSS Redemption'] + bm_raw_final['Seller Portal Redemption']

In [420]:
round(bm_raw_final['Final_Spends'].sum())

853602249

- Market FC, Alpha FC- MP Spillover, MP FC- MP Spillover are zero's because there is base to bifurcate them

In [421]:
bm_raw_final['Market FC'] = np.nan

In [422]:
bm_raw_final['Alpha FC- MP Spillover'] = np.nan

In [423]:
bm_raw_final['MP FC- MP Spillover'] = np.nan

- Final (Amount)

In [424]:
bm_raw_final['Final'] = bm_raw_final['Final_Spends']

In [425]:
round(bm_raw_final['Final'].sum())

853602249

2. PCA Consumption

In [426]:
bss_pca_df = bss_pca[bss_pca['SC match FCFF'] == True].copy()

In [427]:
bss_pca_final = bss_pca_df[['Campaign ID','Ad Account ID', 'BU',
                            'New Supercat','FCFF Alpha/MP', 'Brand',
                            'Team', 'Store Name','spend']]

In [428]:
bss_pca_final = bss_pca_final.rename(columns={"Campaign ID":"Campaign ID",
                                              "Ad Account ID":"Ad Account ID",
                                              "BU":"BU","New Supercat":"Super category",
                                              "FCFF Alpha/MP":"Alpha/MP",
                                              "Brand":"Brand","Team":"KAM/NKAM",
                                              "Store Name":"Store Name","spend":"Spend"})

- Campaign Type

In [429]:
bss_pca_final['Campaign Type'] = "BRAND_PCA"

In [430]:
bss_pca_final.columns

Index(['Campaign ID', 'Ad Account ID', 'BU', 'Super category', 'Alpha/MP',
       'Brand', 'KAM/NKAM', 'Store Name', 'Spend', 'Campaign Type'],
      dtype='object')

- Free Credit for Brand_PCA

In [431]:
bss_totals = bss_redem.groupby(['campaign_id', 'campaign_type', 'account_id'])['campaign_redeem_fc'].sum()
bss_totals.index.names = ['campaign_id', 'campaign_type', 'advertiser_id']

bss_pca_map = bss_pca_final.set_index(['Campaign ID', 'Campaign Type', 'Ad Account ID']).index
bss_pca_final['BSS_Campaign_redeem'] = bss_pca_map.map(bss_totals).fillna(0)


denominator = bss_pca_final.groupby(['Campaign ID', 'Campaign Type', 'Ad Account ID'])['Spend'].transform('sum')

bss_pca_final['Free Credit BPCA'] = np.where(
    (bss_pca_final['Campaign Type'] == "BRAND_PCA") & (denominator > 0),
    (bss_pca_final['BSS_Campaign_redeem'] * bss_pca_final['Spend']) / denominator,
    0
)


In [432]:
round(bss_pca_final['Free Credit BPCA'].sum())

13471688

- OverBurn for Brand_PCA

In [433]:
overburn_totals = mtd_overburn.groupby(['CampaignId', 'CampaignType'])['Overburn'].sum()

bss_pca_final['mapped_Overburn'] = (
    bss_pca_final.set_index(['Campaign ID', 'Campaign Type'])
    .index.map(overburn_totals)
    .fillna(0)
)

denominator = bss_pca_final.groupby(['Campaign ID', 'Campaign Type'])['Spend'].transform('sum')

bss_pca_final['Overburn_BPLA'] = np.where(
    (bss_pca_final['Campaign ID'] == "BRAND_PCA") & (denominator > 0),
    (bss_pca_final['mapped_Overburn'] * bss_pca_final['Spend']) / denominator,
    0
)


In [434]:
round(bss_pca_final['Overburn_BPLA'].sum())

0

- Marketing FC

In [435]:
bss_pca_final['Marketing FC'] = np.nan

- PCA Redemption

In [436]:
bss_totals = bss_redem.groupby(['campaign_id', 'campaign_type', 'account_id'])['campaign_redeem'].sum()
bss_totals.index.names = ['campaign_id', 'campaign_type', 'advertiser_id']

bss_pca_map = bss_pca_final.set_index(['Campaign ID', 'Campaign Type', 'Ad Account ID']).index
bss_pca_final['BSS_Campaign_redeem'] = bss_pca_map.map(bss_totals).fillna(0)

denominator = bss_pca_final.groupby(['Campaign ID', 'Campaign Type', 'Ad Account ID'])['Spend'].transform('sum')

bss_pca_final['PCA Redemption'] = np.where(
    (bss_pca_final['Campaign Type'] == "BRAND_PCA") & (denominator > 0),
    (bss_pca_final['BSS_Campaign_redeem'] * bss_pca_final['Spend']) / denominator,
    0
)


In [437]:
round(bss_pca_final['PCA Redemption'].sum())

72002704

- Net Spend

In [438]:
bss_pca_final['Net Spend'] = bss_pca_final['Spend'] - bss_pca_final['Free Credit BPCA'] - bss_pca_final['Overburn_BPLA']

In [439]:
round(bss_pca_final['Net Spend'].sum())

84915085

- Final Spend

In [440]:
bss_pca_final['Final Spend'] = bss_pca_final['PCA Redemption']

In [441]:
round(bss_pca_final['Final Spend'].sum())

72002704

In [444]:
input_sheet.columns

Index(['Category - BM Raw', 'Supercat Tagging'], dtype='object')

- New Supercategory

In [448]:
map_cat2sc = dict(zip(input_sheet['Category - BM Raw'], input_sheet['Supercat Tagging']))

map_sc2sc = dict(zip(input_sheet['Supercat Tagging'], input_sheet['Supercat Tagging']))

conditions = [
    (bss_pca_final['BU'] == 'BGM') & (bss_pca_final['Super category'] == 'PersonalHealthCare')
]

lookup_result = bss_pca_final['Super category'].map(map_cat2sc).fillna(bss_pca_final['Super category'].map(map_sc2sc)).fillna(bss_pca_final['Super category'])

choices = ["Grooming"]

bss_pca_final['New Super_Category'] = np.select(conditions, choices, default=lookup_result)
